In [1]:
import tqsdk
import pandas as pd
import numpy as np

print("TqSdk:", tqsdk.__version__)
print("pandas:", pd.__version__)
print("numpy:", np.__version__)

TqSdk: 3.10.2
pandas: 3.0.5
numpy: 2.5.3


在使用天勤量化之前，默认您已经知晓并同意以下免责条款，如果不同意请立即停止使用：https://www.shinnytech.com/blog/disclaimer/


In [2]:
import asyncio
import json
from concurrent.futures import ThreadPoolExecutor
from getpass import getpass
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from tqsdk import TqApi, TqAuth, TqSim


# ============================================================
# 1. 研究品种
# ============================================================

SYMBOLS = {
    "M":  "KQ.m@DCE.m",
    "I":  "KQ.m@DCE.i", 
    "RB": "KQ.m@SHFE.rb",
    "CF": "KQ.m@CZCE.CF",
    "SR": "KQ.m@CZCE.SR",
    "V":  "KQ.m@DCE.v",
}

START_DATE = pd.Timestamp("2016-08-01")

PRICE_COLS = ["open", "high", "low", "close"]
VALUE_COLS = PRICE_COLS + ["volume", "oi", "open_oi"]


# ============================================================
# 2. 截止日
# ============================================================

FETCH_TIME = pd.Timestamp.now(tz="Asia/Shanghai")
TODAY = FETCH_TIME.tz_localize(None).normalize()

# 15:30 后才允许使用当天日线
CANDIDATE_END = (
    TODAY
    if FETCH_TIME.hour * 60 + FETCH_TIME.minute >= 15 * 60 + 30
    else TODAY - pd.Timedelta(days=1)
)


# ============================================================
# 3. 输出目录
# ============================================================

OUT = (
    Path.home()
    / "tq_daily_data"
    / FETCH_TIME.strftime("main10_%Y%m%d_%H%M%S")
)

(OUT / "continuous").mkdir(parents=True, exist_ok=True)
(OUT / "contracts").mkdir(parents=True, exist_ok=True)


# ============================================================
# 4. 登录
# ============================================================

TQ_USER = input("天勤账户：").strip()
TQ_PASSWORD = getpass("天勤密码：")

print("研究起点：", START_DATE.date())
print("候选截止：", CANDIDATE_END.date())
print("输出目录：", OUT)

天勤账户： 18757528288
天勤密码： ········


研究起点： 2016-08-01
候选截止： 2026-09-11
输出目录： /home/zilinm2/tq_daily_data/main10_20260912_041428


In [3]:
def save_csv(df, path):
    df.to_csv(path, index=False, encoding="utf-8-sig")


def normalize_dates(values):
    s = pd.to_datetime(values)

    if s.dt.tz is not None:
        s = (
            s.dt
            .tz_convert("Asia/Shanghai")
            .dt.tz_localize(None)
        )

    return s.dt.normalize()


def fetch_daily(api, symbol, data_length, start_date, end_date):
    """
    下载单个合约/主连日线，并截取指定日期。
    """
    serial = api.get_kline_serial(
        symbol,
        duration_seconds=86400,
        data_length=data_length,
    )

    while not api.is_serial_ready(serial):
        api.wait_update()

    df = serial.copy()

    df = df.loc[
        (df["id"] >= 0)
        & (df["datetime"] > 0)
    ].copy()

    if df.empty:
        raise ValueError(f"{symbol} 未返回有效日线")

    df["datetime_nano"] = df["datetime"].astype("int64")

    df["date"] = (
        pd.to_datetime(
            df["datetime_nano"],
            unit="ns",
            utc=True,
        )
        .dt.tz_convert("Asia/Shanghai")
        .dt.tz_localize(None)
        .dt.normalize()
    )

    df = df.rename(columns={"close_oi": "oi"})

    df = df.loc[
        df["date"].between(start_date, end_date)
    ].copy()

    if df["date"].duplicated().any():
        raise RuntimeError(f"{symbol} 存在重复交易日")

    df["symbol"] = symbol

    for col in VALUE_COLS:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    return (
        df[
            ["date", "symbol"]
            + VALUE_COLS
            + ["datetime_nano"]
        ]
        .sort_values("date")
        .reset_index(drop=True)
    )


def download_all():

    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)

    api = None
    logs = []

    continuous_frames = []
    contract_internal_frames = []

    try:

        api = TqApi(
            account=TqSim(),
            auth=TqAuth(TQ_USER, TQ_PASSWORD),
            loop=loop,
        )

        # ====================================================
        # A. 交易日历
        # ====================================================

        calendar = api.get_trading_calendar(
            start_dt=(START_DATE - pd.Timedelta(days=10)).date(),
            end_dt=CANDIDATE_END.date(),
        ).copy()

        calendar["date"] = normalize_dates(calendar["date"])

        trading_dates = pd.DatetimeIndex(
            calendar.loc[calendar["trading"], "date"]
        ).sort_values()

        research_dates = trading_dates[
            trading_dates >= START_DATE
        ]

        if len(research_dates) == 0:
            raise RuntimeError("2016-08-01 之后没有交易日")

        END_DATE = research_dates[-1]

        # ----------------------------------------------------
        # 仅取得研究起点前 1 个交易日
        # 用来判断 2016-08-01 本身是否发生换月
        # 不作为研究数据保存
        # ----------------------------------------------------

        previous_dates = trading_dates[
            trading_dates < START_DATE
        ]

        if len(previous_dates) == 0:
            raise RuntimeError("无法取得起点前一个交易日")

        BOUNDARY_DATE = previous_dates[-1]

        research_dates = research_dates[
            research_dates <= END_DATE
        ]

        print("实际截止日：", END_DATE.date())
        print("边界映射日：", BOUNDARY_DATE.date())


        # ====================================================
        # B. 历史主力映射
        # ====================================================

        print("\n下载历史主力映射……")

        # 不再固定 4000。
        # 所需研究交易日 + 少量 buffer 即可。
        MAP_N = len(research_dates) + 10

        wide = api.query_his_cont_quotes(
            symbol=list(SYMBOLS.values()),
            n=MAP_N,
        ).copy()

        wide["date"] = normalize_dates(wide["date"])

        # ----------------------------------------------------
        # 关键：
        # 第一时间截断。
        # 2015 年及更早映射不会进入后续 contract list。
        # ----------------------------------------------------

        wide = wide.loc[
            wide["date"].between(
                BOUNDARY_DATE,
                END_DATE,
            )
        ].copy()

        expected_dates = trading_dates[
            (trading_dates >= BOUNDARY_DATE)
            & (trading_dates <= END_DATE)
        ]

        missing_dates = expected_dates.difference(
            pd.DatetimeIndex(wide["date"])
        )

        missing_columns = (
            set(SYMBOLS.values())
            - set(wide.columns)
        )

        if len(missing_dates) or missing_columns:
            raise RuntimeError(
                f"历史主力映射不完整："
                f"缺少 {len(missing_dates)} 个交易日；"
                f"缺少列 {sorted(missing_columns)}"
            )

        wide = (
            wide
            .set_index("date")
            .loc[
                expected_dates,
                list(SYMBOLS.values())
            ]
        )

        if (
            wide.isna().any().any()
            or wide.astype(str)
            .apply(lambda s: s.str.strip().eq(""))
            .any().any()
        ):
            raise RuntimeError(
                "研究区间存在空主力映射"
            )


        # ====================================================
        # C. 转成长表，并识别换月
        # ====================================================

        reverse_symbols = {
            v: k for k, v in SYMBOLS.items()
        }

        mapping_full = (
            wide
            .reset_index()
            .melt(
                id_vars="date",
                var_name="continuous_symbol",
                value_name="contract",
            )
        )

        mapping_full["code"] = (
            mapping_full["continuous_symbol"]
            .map(reverse_symbols)
        )

        mapping_full = (
            mapping_full
            .sort_values(["code", "date"])
            .reset_index(drop=True)
        )

        mapping_full["prev_date"] = (
            mapping_full
            .groupby("code")["date"]
            .shift(1)
        )

        mapping_full["old_contract"] = (
            mapping_full
            .groupby("code")["contract"]
            .shift(1)
        )

        mapping_full["roll_flag"] = (
            mapping_full["old_contract"].notna()
            & mapping_full["contract"].ne(
                mapping_full["old_contract"]
            )
        )

        # 正式研究数据严格从 2016-08-01 开始
        mapping = mapping_full.loc[
            mapping_full["date"] >= START_DATE
        ].copy()

        save_csv(
            mapping,
            OUT / "main_mapping.csv",
        )


        # ====================================================
        # D. 下载 10 个主连
        # ====================================================

        MAIN_N = min(
            len(research_dates) + 100,
            10000,
        )

        print("\n下载 6 个主连……")

        for i, (code, symbol) in enumerate(
            SYMBOLS.items(), 1
        ):

            print(
                f"主连 [{i}/6] {code}: {symbol}",
                flush=True,
            )

            try:

                df = fetch_daily(
                    api,
                    symbol,
                    MAIN_N,
                    START_DATE,
                    END_DATE,
                )

                df["code"] = code

                continuous_frames.append(df)

                save_csv(
                    df,
                    OUT / "continuous" / f"{code}.csv",
                )

                logs.append({
                    "kind": "continuous",
                    "symbol": symbol,
                    "status": "downloaded",
                    "rows": len(df),
                    "error": "",
                })

            except Exception as exc:

                logs.append({
                    "kind": "continuous",
                    "symbol": symbol,
                    "status": "failed",
                    "rows": 0,
                    "error": str(exc),
                })

                print("失败：", exc)

            save_csv(
                pd.DataFrame(logs),
                OUT / "download_log.csv",
            )


        # ====================================================
        # E. 只下载 2016-08-01 以后真正当过主力的月合约
        # ====================================================

        contracts = set(
            mapping["contract"].dropna()
        )

        # 如果恰好 2016-08-01 换月，
        # 还需要旧合约用于识别该次切换。
        contracts |= set(
            mapping.loc[
                mapping["roll_flag"],
                "old_contract",
            ].dropna()
        )

        contracts = sorted(contracts)

        print(
            f"\n需要下载实际月合约：{len(contracts)} 个"
        )

        # 月合约生命周期远短于主连。
        # 3000 日线已经非常充足。
        CONTRACT_N = 3000

        for i, contract in enumerate(contracts, 1):

            print(
                f"月合约 [{i}/{len(contracts)}] "
                f"{contract}",
                flush=True,
            )

            try:

                # 内部额外保留前一个交易日，
                # 仅用于 roll event 的 previous-day quote。
                df = fetch_daily(
                    api,
                    contract,
                    CONTRACT_N,
                    BOUNDARY_DATE,
                    END_DATE,
                )

                contract_internal_frames.append(df)

                # 保存到硬盘的数据严格从 2016-08-01 开始
                save_df = df.loc[
                    df["date"] >= START_DATE
                ].copy()

                save_csv(
                    save_df,
                    OUT
                    / "contracts"
                    / f"{contract.replace('.', '_')}.csv",
                )

                logs.append({
                    "kind": "contract",
                    "symbol": contract,
                    "status": "downloaded",
                    "rows": len(save_df),
                    "error": "",
                })

            except Exception as exc:

                logs.append({
                    "kind": "contract",
                    "symbol": contract,
                    "status": "failed",
                    "rows": 0,
                    "error": str(exc),
                })

                print("失败：", exc)

            save_csv(
                pd.DataFrame(logs),
                OUT / "download_log.csv",
            )


        # ====================================================
        # F. 合并
        # ====================================================

        if not continuous_frames:
            raise RuntimeError("6 个主连全部下载失败")

        if not contract_internal_frames:
            raise RuntimeError("实际月合约全部下载失败")

        continuous = pd.concat(
            continuous_frames,
            ignore_index=True,
        )

        actual_internal = pd.concat(
            contract_internal_frames,
            ignore_index=True,
        )

        actual_research = actual_internal.loc[
            actual_internal["date"] >= START_DATE
        ].copy()

        save_csv(
            continuous,
            OUT / "continuous_raw.csv",
        )

        save_csv(
            actual_research,
            OUT / "actual_contracts_raw.csv",
        )

        save_csv(
            calendar.loc[
                calendar["date"] >= START_DATE
            ],
            OUT / "trading_calendar.csv",
        )

        return (
            END_DATE,
            BOUNDARY_DATE,
            research_dates,
            mapping,
            continuous,
            actual_internal,
            actual_research,
            pd.DataFrame(logs),
        )

    finally:

        try:
            if api is not None:
                api.close()
        finally:
            if not loop.is_closed():
                loop.close()

            asyncio.set_event_loop(None)


with ThreadPoolExecutor(max_workers=1) as executor:

    (
        END_DATE,
        BOUNDARY_DATE,
        research_dates,
        mapping,
        continuous,
        actual_internal,
        actual_research,
        download_log,
    ) = executor.submit(download_all).result()


print("\n下载完成。")

display(
    download_log
    .groupby(["kind", "status"])
    .size()
    .rename("数量")
)

实际截止日： 2026-09-11
边界映射日： 2016-07-29

下载历史主力映射……

下载 6 个主连……
主连 [1/6] M: KQ.m@DCE.m
2026-09-12 04:14:57 -     INFO - 通知 : 与 wss://free-api.shinnytech.com/t/nfmd/front/mobile 的网络连接已建立
主连 [2/6] I: KQ.m@DCE.i
主连 [3/6] RB: KQ.m@SHFE.rb
主连 [4/6] CF: KQ.m@CZCE.CF
主连 [5/6] SR: KQ.m@CZCE.SR
主连 [6/6] V: KQ.m@DCE.v

需要下载实际月合约：190 个
月合约 [1/190] CZCE.CF001
月合约 [2/190] CZCE.CF005
月合约 [3/190] CZCE.CF009
月合约 [4/190] CZCE.CF101
月合约 [5/190] CZCE.CF105
月合约 [6/190] CZCE.CF109
月合约 [7/190] CZCE.CF201
月合约 [8/190] CZCE.CF205
月合约 [9/190] CZCE.CF209
月合约 [10/190] CZCE.CF301
月合约 [11/190] CZCE.CF305
月合约 [12/190] CZCE.CF309
月合约 [13/190] CZCE.CF401
月合约 [14/190] CZCE.CF405
月合约 [15/190] CZCE.CF409
月合约 [16/190] CZCE.CF501
月合约 [17/190] CZCE.CF505
月合约 [18/190] CZCE.CF509
月合约 [19/190] CZCE.CF601
月合约 [20/190] CZCE.CF605
月合约 [21/190] CZCE.CF609
月合约 [22/190] CZCE.CF701
月合约 [23/190] CZCE.CF705
月合约 [24/190] CZCE.CF709
月合约 [25/190] CZCE.CF801
月合约 [26/190] CZCE.CF805
月合约 [27/190] CZCE.CF809
月合约 [28/190] CZCE.CF901
月合约 [29/190] C

kind        status    
continuous  downloaded      6
contract    downloaded    190
Name: 数量, dtype: int64

In [6]:
# ============================================================
# 板块 4
# 构造每日历史主力数据 + 换月事件 + 合约生命周期 + 质量检查
#
# 核心原则：
#
# 1. query_his_cont_quotes() 的 mapping
#    决定每个交易日对应的实际主力合约
#
# 2. 实际月合约 OHLCV/OI
#    是后续 QuantStrat 的 canonical / 正式研究数据
#
# 3. KQ.m 主连只作为辅助诊断数据
#    不要求其 OHLC 与实际月合约逐字段完全一致
#
# 4. 不做任何复权、不计算连续收益率
#    所有换月原始信息完整保留
# ============================================================


# ============================================================
# 1. 准备实际月合约原始数据
# ============================================================

actual_table = actual_internal.rename(
    columns={"symbol": "contract"}
).copy()

actual_table = (
    actual_table
    .sort_values(["contract", "date"])
    .reset_index(drop=True)
)

if actual_table.duplicated(
    ["date", "contract"]
).any():
    raise RuntimeError(
        "实际月合约数据存在重复的 date-contract 键"
    )


# ============================================================
# 2. 根据历史 mapping 构造真正的每日主力数据
#
# mapping:
# date + code -> contract
#
# 然后从 actual_table 取得该实际合约当天的 OHLCV/OI
# ============================================================

daily_main = mapping[
    [
        "date",
        "prev_date",
        "code",
        "continuous_symbol",
        "contract",
        "old_contract",
        "roll_flag",
    ]
].copy()


daily_main = daily_main.merge(
    actual_table[
        [
            "date",
            "contract",
            "open",
            "high",
            "low",
            "close",
            "volume",
            "oi",
            "open_oi",
            "datetime_nano",
        ]
    ],
    on=["date", "contract"],
    how="left",
    validate="many_to_one",
)


# ============================================================
# 3. 正式日线数据完整性检查
# ============================================================

price_array = daily_main[
    ["open", "high", "low", "close"]
].to_numpy(dtype=float)

o = price_array[:, 0]
h = price_array[:, 1]
l = price_array[:, 2]
c = price_array[:, 3]


daily_main["ohlc_valid"] = (
    np.isfinite(price_array).all(axis=1)
    & (price_array > 0).all(axis=1)
    & (h >= np.maximum(o, c))
    & (l <= np.minimum(o, c))
    & (h >= l)
)


activity_array = daily_main[
    ["volume", "oi"]
].to_numpy(dtype=float)

daily_main["activity_valid"] = (
    np.isfinite(activity_array).all(axis=1)
    & (activity_array >= 0).all(axis=1)
)


daily_main["data_complete"] = (
    daily_main[
        [
            "open",
            "high",
            "low",
            "close",
            "volume",
            "oi",
        ]
    ]
    .notna()
    .all(axis=1)
)


# 零成交单独记录，但不直接认为 mapping 错误
daily_main["zero_volume"] = (
    daily_main["volume"].eq(0)
)


# ============================================================
# 4. 加入 KQ.m 主连数据
#
# 仅用于辅助诊断。
# 不参与正式数据 ready 判断。
# ============================================================

continuous_check = continuous[
    [
        "date",
        "code",
        "open",
        "high",
        "low",
        "close",
        "volume",
        "oi",
    ]
].rename(
    columns={
        "open": "kqm_open",
        "high": "kqm_high",
        "low": "kqm_low",
        "close": "kqm_close",
        "volume": "kqm_volume",
        "oi": "kqm_oi",
    }
)


daily_main = daily_main.merge(
    continuous_check,
    on=["date", "code"],
    how="left",
    validate="one_to_one",
)


# ============================================================
# 5. 分字段检查 KQ.m 与实际主力合约价格
#
# 注意：
# mismatch 只作为诊断信息，
# 不代表 actual contract mapping 错误。
# ============================================================

for field in ["open", "high", "low", "close"]:

    actual_values = pd.to_numeric(
        daily_main[field],
        errors="coerce",
    )

    kqm_values = pd.to_numeric(
        daily_main[f"kqm_{field}"],
        errors="coerce",
    )

    daily_main[f"kqm_{field}_match"] = (
        actual_values.notna()
        & kqm_values.notna()
        & np.isclose(
            actual_values,
            kqm_values,
            rtol=1e-9,
            atol=1e-8,
        )
    )


# 四个字段全部相同时才 True
# 仅辅助观察，不用于正式数据质量判断
daily_main["kqm_all_ohlc_match"] = (
    daily_main[
        [
            "kqm_open_match",
            "kqm_high_match",
            "kqm_low_match",
            "kqm_close_match",
        ]
    ]
    .all(axis=1)
)


# ============================================================
# 6. 保存正式每日主力数据
# ============================================================

save_csv(
    daily_main,
    OUT / "daily_main_panel.csv",
)


# ============================================================
# 7. 构造主力合约生命周期
#
# 不预设：
# CF = 1/5/9
# M = 1/5/9
# RB = 某几个月
#
# 完全依据每天历史 mapping 自动识别。
# ============================================================

life = (
    mapping
    .sort_values(["code", "date"])
    .copy()
)


# 当 contract 与上一交易日不同，就进入新的主力阶段
life["life_block"] = (
    life
    .groupby("code")["contract"]
    .transform(
        lambda x: x.ne(x.shift()).cumsum()
    )
)


contract_lifecycle = (
    life
    .groupby(
        [
            "code",
            "life_block",
            "contract",
        ],
        as_index=False,
    )
    .agg(
        main_start=("date", "min"),
        main_end=("date", "max"),
        trading_days=("date", "size"),
    )
    .drop(columns="life_block")
    .sort_values(
        ["code", "main_start"]
    )
    .reset_index(drop=True)
)


save_csv(
    contract_lifecycle,
    OUT / "contract_lifecycle.csv",
)


# ============================================================
# 8. 构造换月事件表
# ============================================================

roll_events = (
    mapping.loc[
        mapping["roll_flag"],
        [
            "date",
            "prev_date",
            "code",
            "continuous_symbol",
            "old_contract",
            "contract",
        ],
    ]
    .rename(
        columns={
            "date": "roll_date",
            "contract": "new_contract",
        }
    )
    .sort_values(
        ["code", "roll_date"]
    )
    .reset_index(drop=True)
)


# ============================================================
# 9. 为每次换月加入四组真实行情
#
# old_prev:
#   T-1 旧主力
#
# new_prev:
#   T-1 新主力
#
# old_roll:
#   T 换月当天旧合约
#
# new_roll:
#   T 换月当天新合约
# ============================================================

SNAPSHOT_COLS = [
    "open",
    "high",
    "low",
    "close",
    "volume",
    "oi",
]


def add_contract_snapshot(
    events,
    date_column,
    contract_column,
    prefix,
):

    snapshot = actual_table[
        [
            "date",
            "contract",
        ]
        + SNAPSHOT_COLS
    ].rename(
        columns={
            "date": date_column,
            "contract": contract_column,
            **{
                field: f"{prefix}_{field}"
                for field in SNAPSHOT_COLS
            },
        }
    )

    return events.merge(
        snapshot,
        on=[
            date_column,
            contract_column,
        ],
        how="left",
        validate="many_to_one",
    )


# ------------------------------------------------------------
# T-1 旧主力
# ------------------------------------------------------------

roll_events = add_contract_snapshot(
    roll_events,
    date_column="prev_date",
    contract_column="old_contract",
    prefix="old_prev",
)


# ------------------------------------------------------------
# T-1 新主力
# ------------------------------------------------------------

roll_events = add_contract_snapshot(
    roll_events,
    date_column="prev_date",
    contract_column="new_contract",
    prefix="new_prev",
)


# ------------------------------------------------------------
# T 日旧主力
# ------------------------------------------------------------

roll_events = add_contract_snapshot(
    roll_events,
    date_column="roll_date",
    contract_column="old_contract",
    prefix="old_roll",
)


# ------------------------------------------------------------
# T 日新主力
# ------------------------------------------------------------

roll_events = add_contract_snapshot(
    roll_events,
    date_column="roll_date",
    contract_column="new_contract",
    prefix="new_roll",
)


# ============================================================
# 10. 计算换月时的原始价差
#
# 这里只描述事实。
# 不复权、不修改任何价格。
# ============================================================

# T-1 两个合约的收盘价差
roll_events["prev_close_gap"] = (
    roll_events["new_prev_close"]
    - roll_events["old_prev_close"]
)


# T-1 两个合约的价格比例
roll_events["prev_close_ratio"] = (
    roll_events["new_prev_close"]
    / roll_events["old_prev_close"]
).where(
    roll_events["old_prev_close"] > 0
)


# 换月当天开盘价差
roll_events["roll_open_gap"] = (
    roll_events["new_roll_open"]
    - roll_events["old_roll_open"]
)


# 换月当天收盘价差
roll_events["roll_close_gap"] = (
    roll_events["new_roll_close"]
    - roll_events["old_roll_close"]
)


# ============================================================
# 11. 换月事件完整性
#
# 一个完整换月事件要求：
#
# T-1 old
# T-1 new
# T   old
# T   new
#
# 四组 OHLCV/OI 都存在
# ============================================================

required_roll_fields = []

for prefix in [
    "old_prev",
    "new_prev",
    "old_roll",
    "new_roll",
]:

    for field in SNAPSHOT_COLS:

        required_roll_fields.append(
            f"{prefix}_{field}"
        )


roll_events["event_data_complete"] = (
    roll_events[
        required_roll_fields
    ]
    .notna()
    .all(axis=1)
)


# ============================================================
# 12. 加入 KQ.m 换月日诊断
#
# 仍然只是辅助信息。
# ============================================================

kqm_roll = daily_main[
    [
        "date",
        "code",
        "kqm_open",
        "kqm_high",
        "kqm_low",
        "kqm_close",
        "kqm_open_match",
        "kqm_high_match",
        "kqm_low_match",
        "kqm_close_match",
        "kqm_all_ohlc_match",
    ]
].rename(
    columns={
        "date": "roll_date",
    }
)


roll_events = roll_events.merge(
    kqm_roll,
    on=[
        "roll_date",
        "code",
    ],
    how="left",
    validate="one_to_one",
)


save_csv(
    roll_events,
    OUT / "roll_events.csv",
)


# ============================================================
# 13. KQ.m mismatch 分字段统计
#
# 用于回答：
# 到底是 open 不一致多，
# 还是 high/low/close 都不一致？
#
# 不参与正式 ready 判断。
# ============================================================

kqm_diagnostics = (
    daily_main
    .groupby("code")
    .agg(

        rows=("date", "size"),

        kqm_missing_rows=(
            "kqm_close",
            lambda x: int(
                x.isna().sum()
            ),
        ),

        open_mismatch=(
            "kqm_open_match",
            lambda x: int(
                (~x).sum()
            ),
        ),

        high_mismatch=(
            "kqm_high_match",
            lambda x: int(
                (~x).sum()
            ),
        ),

        low_mismatch=(
            "kqm_low_match",
            lambda x: int(
                (~x).sum()
            ),
        ),

        close_mismatch=(
            "kqm_close_match",
            lambda x: int(
                (~x).sum()
            ),
        ),

        all_ohlc_mismatch=(
            "kqm_all_ohlc_match",
            lambda x: int(
                (~x).sum()
            ),
        ),

    )
    .reset_index()
)


save_csv(
    kqm_diagnostics,
    OUT / "kqm_diagnostics.csv",
)


# ============================================================
# 14. 正式 Quality Report
#
# 注意：
#
# ready 不再考虑 KQ.m mismatch。
#
# 正式数据是否可用只看：
#
# 1. 每个交易日有 mapping
# 2. mapping 指向的真实合约有数据
# 3. OHLC 合法
# 4. volume / oi 合法
# 5. 所有换月事件数据完整
# ============================================================

quality_rows = []

expected_rows = len(research_dates)


for code in SYMBOLS:

    daily = daily_main.loc[
        daily_main["code"].eq(code)
    ].copy()

    rolls = roll_events.loc[
        roll_events["code"].eq(code)
    ].copy()

    life_code = contract_lifecycle.loc[
        contract_lifecycle["code"].eq(code)
    ]


    missing_mapping = int(
        daily["contract"]
        .isna()
        .sum()
    )


    missing_actual = int(
        (~daily["data_complete"])
        .sum()
    )


    invalid_ohlc = int(
        (~daily["ohlc_valid"])
        .sum()
    )


    invalid_activity = int(
        (~daily["activity_valid"])
        .sum()
    )


    incomplete_rolls = int(
        (~rolls["event_data_complete"])
        .sum()
    )


    ready = bool(
        len(daily) == expected_rows
        and missing_mapping == 0
        and missing_actual == 0
        and invalid_ohlc == 0
        and invalid_activity == 0
        and incomplete_rolls == 0
    )


    quality_rows.append({

        "code":
            code,

        "rows":
            len(daily),

        "expected_rows":
            expected_rows,

        "first_date":
            daily["date"].min(),

        "last_date":
            daily["date"].max(),

        "main_contract_periods":
            len(life_code),

        "roll_count":
            len(rolls),

        "missing_contract_mapping":
            missing_mapping,

        "missing_actual_data":
            missing_actual,

        "invalid_ohlc":
            invalid_ohlc,

        "invalid_activity":
            invalid_activity,

        "zero_volume_rows":
            int(
                daily["zero_volume"].sum()
            ),

        "incomplete_roll_events":
            incomplete_rolls,

        # KQ.m 只放诊断数字
        "kqm_all_ohlc_mismatch":
            int(
                (~daily["kqm_all_ohlc_match"])
                .sum()
            ),

        "ready":
            ready,
    })


quality = pd.DataFrame(
    quality_rows
)


save_csv(
    quality,
    OUT / "quality_report.csv",
)


# ============================================================
# 15. 保存异常数据
#
# 只保存真正影响正式数据的问题。
# KQ.m mismatch 不放入正式 issue。
# ============================================================

data_issues = daily_main.loc[
    (~daily_main["data_complete"])
    | (~daily_main["ohlc_valid"])
    | (~daily_main["activity_valid"]),
    [
        "date",
        "code",
        "contract",
        "old_contract",
        "roll_flag",
        "open",
        "high",
        "low",
        "close",
        "volume",
        "oi",
        "data_complete",
        "ohlc_valid",
        "activity_valid",
    ],
].copy()


save_csv(
    data_issues,
    OUT / "data_issues.csv",
)


# ============================================================
# 16. 单独保存 KQ.m mismatch 行
#
# 方便后续研究差异来源，
# 但与正式数据质量完全分离。
# ============================================================

kqm_mismatch_rows = daily_main.loc[
    ~daily_main["kqm_all_ohlc_match"],
    [
        "date",
        "code",
        "contract",
        "roll_flag",

        "open",
        "high",
        "low",
        "close",

        "kqm_open",
        "kqm_high",
        "kqm_low",
        "kqm_close",

        "kqm_open_match",
        "kqm_high_match",
        "kqm_low_match",
        "kqm_close_match",
    ],
].copy()


save_csv(
    kqm_mismatch_rows,
    OUT / "kqm_mismatch_rows.csv",
)


# ============================================================
# 17. Metadata
# ============================================================

metadata = {

    "research_start":
        str(START_DATE.date()),

    "research_end":
        str(END_DATE.date()),

    "symbols":
        SYMBOLS,

    "canonical_contract_selection":
        (
            "historical daily main-contract mapping "
            "from query_his_cont_quotes"
        ),

    "canonical_price_source":
        (
            "raw OHLCV/OI of the actual contract "
            "identified by the historical mapping"
        ),

    "continuous_kqm_role":
        (
            "reference and diagnostic only; "
            "not used as canonical research price"
        ),

    "roll_definition":
        (
            "contract_t != contract_t_minus_1 "
            "within each code independently"
        ),

    "fixed_roll_calendar":
        False,

    "fixed_delivery_month_rule":
        False,

    "price_adjustment":
        "NONE",

    "continuous_return_stitching":
        "NONE",

    "warmup":
        "NONE",

    "boundary_rule":
        (
            "one trading day before 2016-08-01 "
            "is used only to detect a possible roll "
            "exactly on the first research date"
        ),

    "outputs": {

        "daily_main_panel.csv":
            (
                "canonical daily historical "
                "main-contract OHLCV/OI"
            ),

        "main_mapping.csv":
            (
                "daily historical main-contract mapping"
            ),

        "contract_lifecycle.csv":
            (
                "actual main-contract lifecycle "
                "for every product"
            ),

        "roll_events.csv":
            (
                "all detected roll events with "
                "T-1 and T quotes for old/new contracts"
            ),

        "quality_report.csv":
            (
                "formal canonical-data quality report"
            ),

        "kqm_diagnostics.csv":
            (
                "field-level comparison between "
                "KQ.m and mapped actual contract"
            ),

        "kqm_mismatch_rows.csv":
            (
                "raw KQ.m mismatch records "
                "for diagnostic purposes only"
            ),
    },
}


(
    OUT / "metadata.json"
).write_text(
    json.dumps(
        metadata,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)


# ============================================================
# 18. 输出核心结果
# ============================================================

print("=" * 70)
print("板块 4 完成")
print("=" * 70)

print(
    "\n研究区间：",
    START_DATE.date(),
    "→",
    END_DATE.date(),
)

print(
    "研究交易日数量：",
    expected_rows,
)

print(
    "总换月次数：",
    len(roll_events),
)

print(
    "完整换月事件：",
    int(
        roll_events[
            "event_data_complete"
        ].sum()
    ),
    "/",
    len(roll_events),
)


# ============================================================
# 19. 正式质量报告
# ============================================================

print("\n正式数据质量：")

display(
    quality
)


# ============================================================
# 20. KQ.m mismatch 分字段诊断
# ============================================================

print(
    "\nKQ.m vs 实际历史主力合约："
    "分字段诊断（仅参考，不影响 ready）"
)

display(
    kqm_diagnostics
)


# ============================================================
# 21. 查看合约生命周期
# ============================================================

print(
    "\n历史主力合约生命周期："
)

display(
    contract_lifecycle.head(40)
)


# ============================================================
# 22. 查看换月事件
# ============================================================

print(
    "\n换月事件示例："
)

display(
    roll_events[
        [
            "roll_date",
            "code",

            "old_contract",
            "new_contract",

            "old_prev_close",
            "new_prev_close",

            "old_roll_open",
            "new_roll_open",

            "old_roll_close",
            "new_roll_close",

            "prev_close_gap",
            "prev_close_ratio",

            "roll_open_gap",
            "roll_close_gap",

            "event_data_complete",
        ]
    ]
    .sort_values(
        ["code", "roll_date"]
    )
    .head(40)
)


# ============================================================
# 23. 真正的数据异常
# ============================================================

if not data_issues.empty:

    print(
        "\n发现正式数据异常："
    )

    display(
        data_issues.head(30)
    )

else:

    print(
        "\n正式 historical mapping + "
        "actual contract 数据未发现异常。"
    )


# ============================================================
# 24. KQ.m mismatch 示例
# ============================================================

if not kqm_mismatch_rows.empty:

    print(
        "\nKQ.m mismatch 示例 "
        "（仅诊断，不代表实际合约数据错误）："
    )

    display(
        kqm_mismatch_rows.head(30)
    )


# ============================================================
# 25. 最终状态
# ============================================================

ready_count = int(
    quality["ready"].sum()
)

print(
    f"\n正式数据检查通过："
    f"{ready_count}/{len(SYMBOLS)}"
)

print(
    "\n全部文件保存在：",
    OUT,
)

板块 4 完成

研究区间： 2016-08-01 → 2026-09-11
研究交易日数量： 2458
总换月次数： 186
完整换月事件： 186 / 186

正式数据质量：


,code,rows,expected_rows,first_date,last_date,main_contract_periods,roll_count,missing_contract_mapping,missing_actual_data,invalid_ohlc,invalid_activity,zero_volume_rows,incomplete_roll_events,kqm_all_ohlc_mismatch,ready
0,M,2458,2458,2016-08-01,2026-09-11,32,31,0,0,0,0,0,0,1911,True
1,I,2458,2458,2016-08-01,2026-09-11,32,31,0,0,0,0,0,0,1863,True
2,RB,2458,2458,2016-08-01,2026-09-11,32,31,0,0,0,0,0,0,1987,True
3,CF,2458,2458,2016-08-01,2026-09-11,31,30,0,0,0,0,0,0,1880,True
4,SR,2458,2458,2016-08-01,2026-09-11,33,32,0,0,0,0,0,0,1959,True
5,V,2458,2458,2016-08-01,2026-09-11,32,31,0,0,0,0,0,0,1893,True



KQ.m vs 实际历史主力合约：分字段诊断（仅参考，不影响 ready）


,code,rows,kqm_missing_rows,open_mismatch,high_mismatch,low_mismatch,close_mismatch,all_ohlc_mismatch
0,CF,2458,0,1879,2,2,1,1880
1,I,2458,0,1863,2,1,1,1863
2,M,2458,0,1911,2,2,1,1911
3,RB,2458,0,1987,1,0,0,1987
4,SR,2458,0,1959,2,2,1,1959
5,V,2458,0,1893,1,0,0,1893



历史主力合约生命周期：


...


换月事件示例：


,roll_date,code,old_contract,new_contract,old_prev_close,new_prev_close,old_roll_open,new_roll_open,old_roll_close,new_roll_close,prev_close_gap,prev_close_ratio,roll_open_gap,roll_close_gap,event_data_complete
0,2016-11-28,CF,CZCE.CF701,CZCE.CF705,15670.0,15785.0,15700.0,15830.0,15995.0,16255.0,115.0,1.007339,130.0,260.0,True
1,2017-03-31,CF,CZCE.CF705,CZCE.CF709,14745.0,15245.0,14770.0,15275.0,14840.0,15350.0,500.0,1.033910,505.0,510.0,True
2,2017-08-03,CF,CZCE.CF709,CZCE.CF801,15020.0,15120.0,15030.0,15110.0,14995.0,15025.0,100.0,1.006658,80.0,30.0,True
3,2017-11-28,CF,CZCE.CF801,CZCE.CF805,15005.0,15155.0,15020.0,15165.0,15050.0,15195.0,150.0,1.009997,145.0,145.0,True
4,2018-03-28,CF,CZCE.CF805,CZCE.CF809,14985.0,15480.0,14990.0,15475.0,14810.0,15290.0,495.0,1.033033,485.0,480.0,True
5,2018-05-23,CF,CZCE.CF809,CZCE.CF901,17085.0,17865.0,17100.0,17905.0,16840.0,17640.0,780.0,1.045654,805.0,800.0,True
6,2018-11-27,CF,CZCE.CF901,CZCE.CF905,14440.0,14970.0,14480.0,14995.0,14465.0,14950.0,530.0,1.036704,515.0,485.0,True
7,2019-04-04,CF,CZCE.CF905,CZCE.CF909,15175.0,15650.0,15200.0,15660.0,15160.0,15630.0,475.0,1.031301,460.0,470.0,True
8,2019-08-09,CF,CZCE.CF909,CZCE.CF001,12230.0,12765.0,12220.0,12775.0,12175.0,12720.0,535.0,1.043745,555.0,545.0,True
9,2019-12-02,CF,CZCE.CF001,CZCE.CF005,12760.0,13245.0,12760.0,13270.0,12615.0,13085.0,485.0,1.038009,510.0,470.0,True



正式 historical mapping + actual contract 数据未发现异常。

KQ.m mismatch 示例 （仅诊断，不代表实际合约数据错误）：


,date,code,contract,roll_flag,open,high,low,close,kqm_open,kqm_high,kqm_low,kqm_close,kqm_open_match,kqm_high_match,kqm_low_match,kqm_close_match
0,2016-08-01,CF,CZCE.CF701,False,14505.0,14920.0,14465.0,14765.0,14535.0,14920.0,14465.0,14765.0,False,True,True,True
1,2016-08-02,CF,CZCE.CF701,False,14710.0,14875.0,14450.0,14700.0,14745.0,14875.0,14450.0,14700.0,False,True,True,True
2,2016-08-03,CF,CZCE.CF701,False,14710.0,14745.0,14320.0,14505.0,14740.0,14745.0,14320.0,14505.0,False,True,True,True
3,2016-08-04,CF,CZCE.CF701,False,14520.0,14760.0,14355.0,14710.0,14525.0,14760.0,14355.0,14710.0,False,True,True,True
4,2016-08-05,CF,CZCE.CF701,False,14680.0,15175.0,14625.0,15155.0,14670.0,15175.0,14625.0,15155.0,False,True,True,True
5,2016-08-08,CF,CZCE.CF701,False,15200.0,15265.0,14750.0,14755.0,15220.0,15265.0,14750.0,14755.0,False,True,True,True
6,2016-08-09,CF,CZCE.CF701,False,14765.0,14865.0,14525.0,14690.0,14775.0,14865.0,14525.0,14690.0,False,True,True,True
7,2016-08-10,CF,CZCE.CF701,False,14585.0,14780.0,14450.0,14745.0,14590.0,14780.0,14450.0,14745.0,False,True,True,True
8,2016-08-11,CF,CZCE.CF701,False,14760.0,14870.0,14660.0,14730.0,14805.0,14870.0,14660.0,14730.0,False,True,True,True
9,2016-08-12,CF,CZCE.CF701,False,14705.0,14915.0,14600.0,14660.0,14690.0,14915.0,14600.0,14660.0,False,True,True,True



正式数据检查通过：6/6

全部文件保存在： /home/zilinm2/tq_daily_data/main10_20260912_041428


In [7]:
# ============================================================
# Cell 5
# 建立同合约 T-1 / T / T+1 数据连接
#
# 输入：
#   daily_main   <- Cell 4
#   actual_table <- Cell 4
#
# 原则：
#   所有收益率都基于真实实际月合约
#   不再使用 KQ.m 价格
# ============================================================


CONT_FIELDS = [
    "open",
    "high",
    "low",
    "close",
    "volume",
    "oi",
    "open_oi",
]


# ============================================================
# 1. 基础表
# ============================================================

continuous_panel = (
    daily_main[
        [
            "date",
            "prev_date",
            "code",
            "continuous_symbol",
            "contract",
            "old_contract",
            "roll_flag",
        ]
        + CONT_FIELDS
    ]
    .sort_values(["code", "date"])
    .reset_index(drop=True)
    .copy()
)


if continuous_panel.duplicated(
    ["code", "date"]
).any():
    raise RuntimeError(
        "continuous_panel 存在重复 code-date"
    )


# ============================================================
# 2. 当前主力合约在 T-1 的行情
#
# 核心：
#
# contract_t 在 prev_date 的行情
#
# 非换月：
#   和昨日主力完全相同
#
# 换月：
#   使用“新主力自己昨天的价格”
# ============================================================

prev_same = actual_table[
    ["date", "contract"] + CONT_FIELDS
].rename(
    columns={
        "date": "prev_date",
        **{
            field: f"prev_same_{field}"
            for field in CONT_FIELDS
        },
    }
)


continuous_panel = continuous_panel.merge(
    prev_same,
    on=["prev_date", "contract"],
    how="left",
    validate="many_to_one",
)


# ============================================================
# 3. T-1 当时真正的主力合约行情
#
# old_contract = contract_{t-1}
#
# 主要用于把：
#
#   市场真实收益
#   与
#   换月期限价差
#
# 精确拆开。
# ============================================================

prev_active = actual_table[
    ["date", "contract"] + CONT_FIELDS
].rename(
    columns={
        "date": "prev_date",
        "contract": "old_contract",
        **{
            field: f"prev_active_{field}"
            for field in CONT_FIELDS
        },
    }
)


continuous_panel = continuous_panel.merge(
    prev_active,
    on=[
        "prev_date",
        "old_contract",
    ],
    how="left",
    validate="many_to_one",
)


# ============================================================
# 4. 下一个交易日
#
# 注意：
# 这里不是 next day's main contract。
#
# 而是：
#
#   今天选中的 contract_t
#   到明天仍然观察同一个实际合约。
#
# 这对于后面的可交易 ML label 非常重要。
# ============================================================

continuous_panel["next_date"] = (
    continuous_panel
    .groupby("code")["date"]
    .shift(-1)
)


next_same = actual_table[
    ["date", "contract"] + CONT_FIELDS
].rename(
    columns={
        "date": "next_date",
        **{
            field: f"next_same_{field}"
            for field in CONT_FIELDS
        },
    }
)


continuous_panel = continuous_panel.merge(
    next_same,
    on=[
        "next_date",
        "contract",
    ],
    how="left",
    validate="many_to_one",
)


# ============================================================
# 5. 核心价格连接检查
# ============================================================

continuous_panel["return_link_ok"] = (
    continuous_panel["close"].gt(0)
    & continuous_panel["prev_same_close"].gt(0)
    & np.isfinite(
        continuous_panel["close"]
    )
    & np.isfinite(
        continuous_panel["prev_same_close"]
    )
)


bad_return_links = continuous_panel.loc[
    ~continuous_panel["return_link_ok"],
    [
        "date",
        "code",
        "contract",
        "old_contract",
        "roll_flag",
        "close",
        "prev_same_close",
    ],
].copy()


if not bad_return_links.empty:

    save_csv(
        bad_return_links,
        OUT / "bad_return_links.csv",
    )

    raise RuntimeError(
        f"发现 {len(bad_return_links)} 个无法构造"
        "同合约连续收益率的交易日。"
        "已保存 bad_return_links.csv；"
        "停止后续处理，禁止自动填充。"
    )


print(
    "Cell 5：同合约价格连接完成。"
)

print(
    "有效连接：",
    len(continuous_panel),
    "/",
    len(continuous_panel),
)

Cell 5：同合约价格连接完成。
有效连接： 14748 / 14748


In [8]:
# ============================================================
# Cell 6
# 连续收益率 + 换月分解 + 因果连续 OHLC
# ============================================================


# ============================================================
# 1. 核心 stitched return
#
# 永远是：
#
# 当天实际主力 close
# --------------------
# 同一个合约 T-1 close
#
# 换月日不跨合约。
# ============================================================

continuous_panel["stitched_return"] = (
    continuous_panel["close"]
    / continuous_panel["prev_same_close"]
    - 1.0
)


continuous_panel["stitched_log_return"] = (
    np.log(
        continuous_panel["close"]
    )
    - np.log(
        continuous_panel["prev_same_close"]
    )
)


# ============================================================
# 2. 从昨日同合约 close 到今天 OHLC
#
# 后面做：
#   gap
#   candle
#   ATR
#   range
#   volatility
# 等特征会用到。
# ============================================================

for field in [
    "open",
    "high",
    "low",
    "close",
]:

    continuous_panel[
        f"{field}_from_prev_return"
    ] = (
        continuous_panel[field]
        / continuous_panel["prev_same_close"]
        - 1.0
    )


    continuous_panel[
        f"{field}_from_prev_log"
    ] = (
        np.log(
            continuous_panel[field]
        )
        - np.log(
            continuous_panel["prev_same_close"]
        )
    )


# ============================================================
# 3. 日内收益
# ============================================================

continuous_panel[
    "intraday_return"
] = (
    continuous_panel["close"]
    / continuous_panel["open"]
    - 1.0
)


continuous_panel[
    "intraday_log_return"
] = (
    np.log(
        continuous_panel["close"]
    )
    - np.log(
        continuous_panel["open"]
    )
)


# ============================================================
# 4. 日内振幅
# ============================================================

continuous_panel[
    "log_high_low_range"
] = (
    np.log(
        continuous_panel["high"]
    )
    - np.log(
        continuous_panel["low"]
    )
)


# ============================================================
# 5. naive 主力拼接收益
#
# 这就是如果直接：
#
# 今天主力 close /
# 昨天主力 close
#
# 会得到的结果。
#
# 换月时包含机械期限价差。
# ============================================================

prev_active_ok = (
    continuous_panel[
        "prev_active_close"
    ].gt(0)
)


continuous_panel[
    "naive_mapped_log_return"
] = (
    np.log(
        continuous_panel["close"]
    )
    - np.log(
        continuous_panel["prev_active_close"]
    )
).where(
    prev_active_ok
)


# ============================================================
# 6. 换月期限结构项
#
# new contract T-1 close
# ----------------------
# old contract T-1 close
#
# 非换月日理论上 = 0
# ============================================================

continuous_panel[
    "roll_basis_log"
] = (
    np.log(
        continuous_panel[
            "prev_same_close"
        ]
    )
    - np.log(
        continuous_panel[
            "prev_active_close"
        ]
    )
).where(
    prev_active_ok
)


continuous_panel[
    "roll_basis_return"
] = (
    continuous_panel[
        "prev_same_close"
    ]
    / continuous_panel[
        "prev_active_close"
    ]
    - 1.0
).where(
    prev_active_ok
)


# ============================================================
# 7. 数学恒等式检查
#
# naive return
#
# =
#
# stitched market return
# +
# roll basis
# ============================================================

continuous_panel[
    "return_decomposition_error"
] = (
    continuous_panel[
        "naive_mapped_log_return"
    ]
    - continuous_panel[
        "stitched_log_return"
    ]
    - continuous_panel[
        "roll_basis_log"
    ]
)


# ============================================================
# 8. 构造因果连续 OHLC Index
#
# 不是传统 backward adjustment。
#
# 不修改过去。
# 不使用未来。
#
# 每个品种在研究起点前一交易日假设 index=100。
#
# 每一天的 scale：
#
# previous linked close
# ---------------------
# 当前合约 T-1 close
#
# 所以换月日也不会出现机械 gap。
# ============================================================

def build_causal_linked_ohlc(group):

    group = (
        group
        .sort_values("date")
        .copy()
    )

    group["linked_prev_close"] = np.nan
    group["linked_factor"] = np.nan

    for field in [
        "open",
        "high",
        "low",
        "close",
    ]:
        group[
            f"linked_{field}"
        ] = np.nan


    # 研究起点前一交易日：
    # 假定连续指数 = 100
    previous_linked_close = 100.0


    for idx in group.index:

        denominator = group.at[
            idx,
            "prev_same_close",
        ]

        if (
            not np.isfinite(denominator)
            or denominator <= 0
        ):
            raise RuntimeError(
                f"{group.at[idx, 'code']} "
                f"{group.at[idx, 'date']} "
                "缺少合法 prev_same_close"
            )


        factor = (
            previous_linked_close
            / denominator
        )


        group.at[
            idx,
            "linked_prev_close",
        ] = previous_linked_close


        group.at[
            idx,
            "linked_factor",
        ] = factor


        for field in [
            "open",
            "high",
            "low",
            "close",
        ]:

            group.at[
                idx,
                f"linked_{field}",
            ] = (
                group.at[idx, field]
                * factor
            )


        previous_linked_close = (
            group.at[
                idx,
                "linked_close",
            ]
        )


    return group


continuous_panel = pd.concat(
    [
        build_causal_linked_ohlc(group)
        for _, group in continuous_panel.groupby(
            "code",
            sort=False,
        )
    ],
    ignore_index=True,
)


continuous_panel = (
    continuous_panel
    .sort_values(
        ["code", "date"]
    )
    .reset_index(drop=True)
)


# ============================================================
# 9. 连续指数自身收益验证
# ============================================================

continuous_panel[
    "linked_log_return"
] = (
    np.log(
        continuous_panel[
            "linked_close"
        ]
    )
    - np.log(
        continuous_panel[
            "linked_prev_close"
        ]
    )
)


continuous_panel[
    "linked_return_error"
] = (
    continuous_panel[
        "linked_log_return"
    ]
    - continuous_panel[
        "stitched_log_return"
    ]
)


print(
    "Cell 6：连续收益率及因果连续 OHLC 构造完成。"
)

Cell 6：连续收益率及因果连续 OHLC 构造完成。


In [9]:
# ============================================================
# Cell 7
# Volume/OI + causal roll features + ML/DL targets
# ============================================================


# ============================================================
# 1. Volume / OI 同合约变化
#
# 不能跨 old → new contract 直接比较。
# ============================================================

for field in [
    "volume",
    "oi",
    "open_oi",
]:

    current = pd.to_numeric(
        continuous_panel[field],
        errors="coerce",
    )

    previous = pd.to_numeric(
        continuous_panel[
            f"prev_same_{field}"
        ],
        errors="coerce",
    )


    valid = (
        current.notna()
        & previous.notna()
        & current.ge(0)
        & previous.ge(0)
    )


    continuous_panel[
        f"{field}_same_log_change"
    ] = (
        np.log1p(
            current.where(valid)
        )
        - np.log1p(
            previous.where(valid)
        )
    )


# ============================================================
# 2. Roll flag
# ============================================================

continuous_panel[
    "is_roll"
] = (
    continuous_panel[
        "roll_flag"
    ]
    .astype(int)
)


# ============================================================
# 3. 距最近一次已发生换月多少交易日
#
# 这是 causal feature：
# 只依赖过去。
#
# 研究开始后第一次 roll 之前保持 NaN，
# 因为我们没有向 2015 回溯。
# ============================================================

def add_days_since_roll(group):

    group = (
        group
        .sort_values("date")
        .copy()
    )

    positions = pd.Series(
        np.arange(len(group)),
        index=group.index,
        dtype=float,
    )

    last_roll_position = (
        positions
        .where(
            group["roll_flag"]
        )
        .ffill()
    )

    group[
        "days_since_roll"
    ] = (
        positions
        - last_roll_position
    )

    return group


continuous_panel = pd.concat(
    [
        add_days_since_roll(group)
        for _, group in continuous_panel.groupby(
            "code",
            sort=False,
        )
    ],
    ignore_index=True,
)


continuous_panel = (
    continuous_panel
    .sort_values(
        ["code", "date"]
    )
    .reset_index(drop=True)
)


# ============================================================
# 4. Research-only 下一日主力收益
#
# 这是：
#
# tomorrow's stitched market return
#
# 可以作为研究目标，
# 但它使用 tomorrow 的主力选择结果。
#
# 不应该直接解释为：
# “今天一定能够执行的 PnL”
# ============================================================

continuous_panel[
    "target_next_main_log_return"
] = (
    continuous_panel
    .groupby("code")[
        "stitched_log_return"
    ]
    .shift(-1)
)


continuous_panel[
    "target_next_main_return"
] = (
    continuous_panel
    .groupby("code")[
        "stitched_return"
    ]
    .shift(-1)
)


# ============================================================
# 5. 可执行性更强的 target
#
# 今天 t：
# contract_t 已经确定。
#
# 持有同一个 contract_t 到下一交易日。
#
# 不需要提前知道 c_{t+1} 是谁。
# ============================================================

next_close_valid = (
    continuous_panel[
        "next_same_close"
    ].gt(0)
    & continuous_panel[
        "close"
    ].gt(0)
)


continuous_panel[
    "target_hold_current_log_return"
] = (
    np.log(
        continuous_panel[
            "next_same_close"
        ]
    )
    - np.log(
        continuous_panel[
            "close"
        ]
    )
).where(
    next_close_valid
)


continuous_panel[
    "target_hold_current_return"
] = (
    continuous_panel[
        "next_same_close"
    ]
    / continuous_panel[
        "close"
    ]
    - 1.0
).where(
    next_close_valid
)


# ============================================================
# 6. 如果模型在 t 日收盘后产生信号，
#    下一日开盘执行：
#
# next open → next close
#
# 这是非常重要的现实交易 target。
# ============================================================

next_oc_valid = (
    continuous_panel[
        "next_same_open"
    ].gt(0)
    & continuous_panel[
        "next_same_close"
    ].gt(0)
)


continuous_panel[
    "target_next_open_close_log_return"
] = (
    np.log(
        continuous_panel[
            "next_same_close"
        ]
    )
    - np.log(
        continuous_panel[
            "next_same_open"
        ]
    )
).where(
    next_oc_valid
)


continuous_panel[
    "target_next_open_close_return"
] = (
    continuous_panel[
        "next_same_close"
    ]
    / continuous_panel[
        "next_same_open"
    ]
    - 1.0
).where(
    next_oc_valid
)


# ============================================================
# 7. 隔夜 / 开盘跳空
#
# t close -> t+1 open
# 同一个 contract_t
# ============================================================

continuous_panel[
    "target_next_open_gap_log"
] = (
    np.log(
        continuous_panel[
            "next_same_open"
        ]
    )
    - np.log(
        continuous_panel[
            "close"
        ]
    )
).where(
    next_oc_valid
)


# ============================================================
# 8. 下一日 label 是否有效
#
# 每个品种最后一个交易日自然没有 T+1，
# 所以最后一行 NaN 是正确的。
# ============================================================

continuous_panel[
    "forward_label_available"
] = (
    continuous_panel[
        "next_date"
    ].notna()
    & continuous_panel[
        "next_same_close"
    ].gt(0)
)


print(
    "Cell 7：Volume/OI 和 ML/DL targets 构造完成。"
)

Cell 7：Volume/OI 和 ML/DL targets 构造完成。


In [10]:
# ============================================================
# Cell 8
# 最终数学审计 + 保存最终连续研究数据
# ============================================================


TOL = 1e-10


# ============================================================
# 1. 每个品种质量统计
# ============================================================

audit_rows = []


for code in SYMBOLS:

    g = (
        continuous_panel.loc[
            continuous_panel["code"].eq(code)
        ]
        .sort_values("date")
        .copy()
    )


    rolls = g.loc[
        g["roll_flag"]
    ]


    non_rolls = g.loc[
        ~g["roll_flag"]
    ]


    # -----------------------------------------------
    # A. 分解恒等式
    # -----------------------------------------------

    decomposition_error = (
        g[
            "return_decomposition_error"
        ]
        .dropna()
        .abs()
    )


    max_decomposition_error = (
        decomposition_error.max()
        if len(decomposition_error)
        else np.nan
    )


    # -----------------------------------------------
    # B. linked return 必须等于 stitched return
    # -----------------------------------------------

    linked_error = (
        g[
            "linked_return_error"
        ]
        .dropna()
        .abs()
    )


    max_linked_error = (
        linked_error.max()
        if len(linked_error)
        else np.nan
    )


    # -----------------------------------------------
    # C. 非换月日 roll basis 必须 ≈ 0
    # -----------------------------------------------

    nonroll_basis = (
        non_rolls[
            "roll_basis_log"
        ]
        .dropna()
        .abs()
    )


    max_nonroll_basis = (
        nonroll_basis.max()
        if len(nonroll_basis)
        else np.nan
    )


    # -----------------------------------------------
    # D. OHLC 合法性
    # -----------------------------------------------

    linked_prices = g[
        [
            "linked_open",
            "linked_high",
            "linked_low",
            "linked_close",
        ]
    ].to_numpy(dtype=float)


    lo = linked_prices[:, 0]
    lh = linked_prices[:, 1]
    ll = linked_prices[:, 2]
    lc = linked_prices[:, 3]


    linked_ohlc_valid = (
        np.isfinite(
            linked_prices
        ).all(axis=1)
        & (
            linked_prices > 0
        ).all(axis=1)
        & (
            lh >= np.maximum(
                lo,
                lc,
            )
        )
        & (
            ll <= np.minimum(
                lo,
                lc,
            )
        )
        & (
            lh >= ll
        )
    )


    invalid_linked_ohlc = int(
        (~linked_ohlc_valid).sum()
    )


    # -----------------------------------------------
    # E. forward labels
    #
    # 最后一行没有 label 是正常的。
    # 其他缺失需要检查。
    # -----------------------------------------------

    expected_forward = (
        g["next_date"].notna()
    )


    unexpected_missing_forward = int(
        (
            expected_forward
            & ~g[
                "forward_label_available"
            ]
        ).sum()
    )


    # -----------------------------------------------
    # F. 最终 ready
    # -----------------------------------------------

    ready = bool(

        np.isfinite(
            max_decomposition_error
        )

        and (
            max_decomposition_error
            <= TOL
        )

        and np.isfinite(
            max_linked_error
        )

        and (
            max_linked_error
            <= TOL
        )

        and (
            (
                not np.isfinite(
                    max_nonroll_basis
                )
            )
            or (
                max_nonroll_basis
                <= TOL
            )
        )

        and (
            invalid_linked_ohlc == 0
        )

        and (
            unexpected_missing_forward == 0
        )
    )


    audit_rows.append({

        "code":
            code,

        "rows":
            len(g),

        "roll_count":
            len(rolls),

        "max_decomposition_error":
            max_decomposition_error,

        "max_linked_return_error":
            max_linked_error,

        "max_nonroll_basis":
            max_nonroll_basis,

        "invalid_linked_ohlc":
            invalid_linked_ohlc,

        "unexpected_missing_forward_labels":
            unexpected_missing_forward,

        "ready_for_feature_engineering":
            ready,
    })


return_quality = pd.DataFrame(
    audit_rows
)


# ============================================================
# 2. 如果核心数学关系失败，直接停止
# ============================================================

failed_codes = return_quality.loc[
    ~return_quality[
        "ready_for_feature_engineering"
    ]
]


save_csv(
    return_quality,
    OUT / "return_quality_report.csv",
)


if not failed_codes.empty:

    display(return_quality)

    raise RuntimeError(
        "连续收益率数学审计未通过。"
        "禁止进入后续特征工程 / ML / DL。"
    )


# ============================================================
# 3. 保存完整研究 Panel
# ============================================================

save_csv(
    continuous_panel,
    OUT / "continuous_research_panel.csv",
)


# ============================================================
# 4. 保存紧凑版连续收益率
# ============================================================

continuous_returns = continuous_panel[
    [
        "date",
        "code",
        "contract",
        "old_contract",
        "roll_flag",

        "open",
        "high",
        "low",
        "close",

        "prev_same_close",

        "stitched_return",
        "stitched_log_return",

        "linked_open",
        "linked_high",
        "linked_low",
        "linked_close",

        "volume",
        "oi",

        "volume_same_log_change",
        "oi_same_log_change",

        "roll_basis_return",
        "roll_basis_log",

        "days_since_roll",
    ]
].copy()


save_csv(
    continuous_returns,
    OUT / "continuous_returns.csv",
)


# ============================================================
# 5. 单独保存 ML/DL targets
# ============================================================

ml_targets = continuous_panel[
    [
        "date",
        "code",
        "contract",

        "target_next_main_return",
        "target_next_main_log_return",

        "target_hold_current_return",
        "target_hold_current_log_return",

        "target_next_open_close_return",
        "target_next_open_close_log_return",

        "target_next_open_gap_log",

        "forward_label_available",
    ]
].copy()


save_csv(
    ml_targets,
    OUT / "ml_targets.csv",
)


# ============================================================
# 6. 单独保存 roll audit
#
# 用来人工核验每次换月：
#
# naive return
# =
# real return
# +
# term-structure gap
# ============================================================

roll_return_audit = continuous_panel.loc[
    continuous_panel["roll_flag"],
    [
        "date",
        "code",

        "old_contract",
        "contract",

        "prev_active_close",
        "prev_same_close",
        "close",

        "naive_mapped_log_return",
        "stitched_log_return",
        "roll_basis_log",

        "return_decomposition_error",

        "linked_prev_close",
        "linked_close",
    ],
].copy()


save_csv(
    roll_return_audit,
    OUT / "roll_return_audit.csv",
)


# ============================================================
# 7. 更新 metadata
# ============================================================

metadata_path = (
    OUT / "metadata.json"
)


if metadata_path.exists():

    metadata = json.loads(
        metadata_path.read_text(
            encoding="utf-8"
        )
    )

else:

    metadata = {}


metadata[
    "continuous_return_construction"
] = {

    "method":
        "causal same-contract return stitching",

    "core_formula":
        (
            "return_t = "
            "price(current_main_contract_t, t) / "
            "price(current_main_contract_t, t-1) - 1"
        ),

    "roll_treatment":
        (
            "never calculate return directly from "
            "old contract close to new contract close"
        ),

    "roll_basis":
        (
            "new contract T-1 close / "
            "old contract T-1 close - 1"
        ),

    "linked_ohlc":
        (
            "causal forward-linked research index; "
            "previous history is never rewritten"
        ),

    "linked_index_anchor":
        (
            "100 at the trading day immediately "
            "before the research start"
        ),

    "volume_oi":
        (
            "raw actual-contract values; "
            "changes calculated against the same "
            "contract on the previous trading day"
        ),

    "ml_target_primary_tradeable":
        "target_hold_current_log_return",

    "ml_target_next_open_execution":
        "target_next_open_close_log_return",

    "research_only_market_target":
        "target_next_main_log_return",

    "missing_policy":
        "no filling, no interpolation, fail core audit",

    "future_information":
        (
            "linked prices and stitched returns "
            "use no future prices or future adjustment factors"
        ),
}


metadata_path.write_text(
    json.dumps(
        metadata,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)


# ============================================================
# 8. 最终输出
# ============================================================

print("=" * 75)
print("连续收益率 / 换月处理完成")
print("=" * 75)

display(
    return_quality
)


print(
    "\n换月收益分解示例："
)

display(
    roll_return_audit.head(30)
)


print(
    "\n最终通过：",
    int(
        return_quality[
            "ready_for_feature_engineering"
        ].sum()
    ),
    "/",
    len(SYMBOLS),
)


print(
    "\n核心输出文件："
)

print(
    OUT / "continuous_returns.csv"
)

print(
    OUT / "continuous_research_panel.csv"
)

print(
    OUT / "ml_targets.csv"
)

print(
    OUT / "roll_return_audit.csv"
)

print(
    OUT / "return_quality_report.csv"
)

连续收益率 / 换月处理完成


,code,rows,roll_count,max_decomposition_error,max_linked_return_error,max_nonroll_basis,invalid_linked_ohlc,unexpected_missing_forward_labels,ready_for_feature_engineering
0,M,2458,31,0.0,1.776357e-15,0.0,0,0,True
1,I,2458,31,0.0,8.881784e-16,0.0,0,0,True
2,RB,2458,31,0.0,1.776357e-15,0.0,0,0,True
3,CF,2458,30,0.0,1.776357e-15,0.0,0,0,True
4,SR,2458,32,0.0,1.776357e-15,0.0,0,0,True
5,V,2458,31,0.0,1.776357e-15,0.0,0,0,True



换月收益分解示例：


,date,code,old_contract,contract,prev_active_close,prev_same_close,close,naive_mapped_log_return,stitched_log_return,roll_basis_log,return_decomposition_error,linked_prev_close,linked_close
78,2016-11-28,CF,CZCE.CF701,CZCE.CF705,15670.0,15785.0,16255.0,0.036652,0.029340,0.007312,0.0,107.734617,110.942426
161,2017-03-31,CF,CZCE.CF705,CZCE.CF709,14745.0,15245.0,15350.0,0.040211,0.006864,0.033348,0.0,100.636485,101.329620
245,2017-08-03,CF,CZCE.CF709,CZCE.CF801,15020.0,15120.0,15025.0,0.000333,-0.006303,0.006636,0.0,99.151198,98.528224
323,2017-11-28,CF,CZCE.CF801,CZCE.CF805,15005.0,15155.0,15195.0,0.012583,0.002636,0.009947,0.0,98.397072,98.656780
403,2018-03-28,CF,CZCE.CF805,CZCE.CF809,14985.0,15480.0,15290.0,0.020149,-0.012350,0.032499,0.0,97.293310,96.099142
439,2018-05-23,CF,CZCE.CF809,CZCE.CF901,17085.0,17865.0,17640.0,0.031968,-0.012674,0.044643,0.0,107.380892,106.028488
566,2018-11-27,CF,CZCE.CF901,CZCE.CF905,14440.0,14970.0,14950.0,0.034709,-0.001337,0.036046,0.0,86.794295,86.678337
651,2019-04-04,CF,CZCE.CF905,CZCE.CF909,15175.0,15650.0,15630.0,0.029543,-0.001279,0.030822,0.0,87.982861,87.870423
737,2019-08-09,CF,CZCE.CF909,CZCE.CF001,12230.0,12765.0,12720.0,0.039284,-0.003531,0.042815,0.0,68.755935,68.513553
812,2019-12-02,CF,CZCE.CF001,CZCE.CF005,12760.0,13245.0,13085.0,0.025151,-0.012154,0.037305,0.0,68.729004,67.898756



最终通过： 6 / 6

核心输出文件：
/home/zilinm2/tq_daily_data/main10_20260912_041428/continuous_returns.csv
/home/zilinm2/tq_daily_data/main10_20260912_041428/continuous_research_panel.csv
/home/zilinm2/tq_daily_data/main10_20260912_041428/ml_targets.csv
/home/zilinm2/tq_daily_data/main10_20260912_041428/roll_return_audit.csv
/home/zilinm2/tq_daily_data/main10_20260912_041428/return_quality_report.csv
